In [1]:
import pandas as pd
df = pd.read_csv("../data/raw/ChargingRecords.csv")

In [2]:
df["StartDatetime"] = pd.to_datetime(df["StartDatetime"], errors="coerce")
df["EndDatetime"] = pd.to_datetime(df["EndDatetime"], errors="coerce")
df["StartDay"] = pd.to_datetime(df["StartDay"], errors="coerce")
df["EndDay"] = pd.to_datetime(df["EndDay"], errors="coerce")

# these are IDs/categories, not numbers to do math on
for col in ["UserID", "ChargerID", "ChargerCompany", "ChargerType"]:
    df[col] = df[col].astype("category")

print(df.dtypes)
print(df[["StartDatetime", "EndDatetime"]].isnull().sum())

UserID                  category
ChargerID               category
ChargerCompany          category
Location                  object
ChargerType             category
StartDay          datetime64[ns]
StartTime                 object
EndDay            datetime64[ns]
EndTime                   object
StartDatetime     datetime64[ns]
EndDatetime       datetime64[ns]
Duration                   int64
Demand                   float64
dtype: object
StartDatetime    0
EndDatetime      0
dtype: int64


In [3]:
print((df.isnull().sum() / len(df) * 100).round(2))
print("Duplicates:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

UserID            0.0
ChargerID         0.0
ChargerCompany    0.0
Location          0.0
ChargerType       0.0
StartDay          0.0
StartTime         0.0
EndDay            0.0
EndTime           0.0
StartDatetime     0.0
EndDatetime       0.0
Duration          0.0
Demand            0.0
dtype: float64
Duplicates: 13


In [4]:
print("End before Start:", (df["EndDatetime"] < df["StartDatetime"]).sum())
print("Demand <= 0:", (df["Demand"] <= 0).sum())
print("Duration <= 0:", (df["Duration"] <= 0).sum())
print(df[["Duration", "Demand"]].describe())

# check Duration matches the timestamps (find the unit)
calc_min = (df["EndDatetime"] - df["StartDatetime"]).dt.total_seconds() / 60
print(calc_min.head(), df["Duration"].head())

End before Start: 17
Demand <= 0: 0
Duration <= 0: 735
           Duration        Demand
count  72843.000000  72843.000000
mean     151.183998     17.442667
std      146.420312     13.491446
min      -29.000000      0.010000
25%       40.000000      7.530000
50%      112.000000     14.100000
75%      202.000000     23.200000
max     1573.000000     97.000000
0    185.0
1     90.0
2     60.0
3    344.0
4    300.0
dtype: float64 0    185
1     90
2     60
3    344
4    300
Name: Duration, dtype: int64


In [5]:
for col in ["Demand", "Duration"]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((df[col] < low) | (df[col] > high)).sum()
    print(col, "outliers:", n, f"({n/len(df)*100:.1f}%)")

Demand outliers: 3441 (4.7%)
Duration outliers: 3922 (5.4%)


In [7]:
import os

os.makedirs("../data/processed", exist_ok=True)
df.to_csv("../data/processed/cleaned_data.csv", index=False)
print(df.shape)

(72843, 13)
